In [ ]:
import torch
import pandas as pd
import numpy as np
from process import create_sequences, BKTSequenceDataset, bkt_collate_fn
from model import NeuralBKT, BKTConfig
from torch.utils.data import DataLoader
import os

def load_model(checkpoint_path, config, device):
    model = NeuralBKT(config).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model

@torch.no_grad()
def extract_bkt_parameters(model, sequences, device, skill_id_to_name=None):
    """
    Extracts BKT parameters for all sequences.
    Returns a list of DataFrames, one per sequence.
    """
    dataset = BKTSequenceDataset(sequences)
    loader = DataLoader(dataset, batch_size=1, collate_fn=bkt_collate_fn)  # batch_size=1 for simplicity
    
    results = []
    
    for idx, (obs, output) in enumerate(loader):
        obs = obs.to(device)
        B, T, D = obs.shape
        
        # Get all BKT parameters
        corrects_pred, latents, adjusted_params, _ = model(
            obs, 
            output=obs,  # dummy output
            return_all=True
        )
        
        # Extract parameters
        l = adjusted_params[0, :, 0].cpu().numpy()  # (T,)
        g = adjusted_params[0, :, 2].cpu().numpy()  # (T,)
        s = adjusted_params[0, :, 3].cpu().numpy()  # (T,)
        
        # Latent knowledge states — convert list of tensors to array
        latent_array = torch.stack(latents, dim=0).cpu().numpy()  # (T, B, n_skills) → (T, 1, n_skills)
        latent_array = latent_array[:, 0, :]  # (T, n_skills)
        
        # Get skill IDs and correctness from obs
        skill_ids = obs[0, :, 1].long().cpu().numpy()  # (T,)
        correctness = obs[0, :, 0].cpu().numpy()       # (T,)
        
        # Create DataFrame
        df_seq = pd.DataFrame({
            'timestep': np.arange(T),
            'skill_id': skill_ids,
            'correct': correctness,
            'predicted_correct': corrects_pred[0].cpu().numpy(),
            'learning_rate': l,
            'guess_rate': g,
            'slip_rate': s,
        })
        
        # Add latent knowledge for each skill
        for skill_idx in range(latent_array.shape[1]):
            skill_name = skill_id_to_name[skill_idx] if skill_id_to_name else f"skill_{skill_idx}"
            df_seq[f'knowledge_{skill_name}'] = latent_array[:, skill_idx]
        
        results.append(df_seq)
    
    return results

if __name__ == '__main__':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Load data
    df = pd.read_csv('data/icecream_3rd.csv')  # or your dataset
    sequences, _ = create_sequences(df, block_size=512)
    
    # Create skill_id to name mapping (if available)
    if 'skill_name' in df.columns:
        skill_map = df[['skill_id', 'skill_name']].drop_duplicates().set_index('skill_id')['skill_name'].to_dict()
    else:
        skill_map = None
    
    # Load model
    N_SKILLS = df['skill_id'].nunique()
    config = BKTConfig(n_skills=N_SKILLS, n_embd=128, n_layer=3, n_head=4, block_size=512)
    model = load_model('./checkpoints/icecream/best_model.pt', config, device)
    
    # Extract parameters
    print("Extracting BKT parameters...")
    results = extract_bkt_parameters(model, sequences, device, skill_id_to_name=skill_map)
    
    # Save results
    os.makedirs('./results', exist_ok=True)
    for i, df_seq in enumerate(results):
        df_seq.to_csv(f'./results/sequence_{i}.csv', index=False)
        print(f"Saved sequence {i} with {len(df_seq)} steps")
    
    print(f"Extracted parameters for {len(results)} sequences. Results saved in ./results/")